In [1]:
import sys
import tensorflow as tf


print("Python version:", sys.version)
print("Python executable:", sys.executable)

print("TensorFlow version:", tf.__version__)

Python version: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
Python executable: C:\Users\HP\miniconda3\envs\dog-vision\python.exe
TensorFlow version: 2.21.0


In [2]:
from datasets import load_dataset


import datasets

print("Datasets library version:", datasets.__version__)

# Load the email spam dataset
email_dataset = load_dataset("diegovelilla/email-classification-dataset")

C:\Users\HP\miniconda3\envs\dog-vision\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Datasets library version: 5.1.0


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


In [3]:
print(email_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8175
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2725
    })
})


In [4]:
train_data = email_dataset["train"]
test_data = email_dataset["test"]

In [5]:
from collections import Counter

print("Training labels:")
print(Counter(train_data["label"]))

print("\nTesting labels:")
print(Counter(test_data["label"]))

Training labels:
Counter({'spam': 4125, 'not_spam': 4050})

Testing labels:
Counter({'spam': 1375, 'not_spam': 1350})


## **A) Duplicate detection and removal**

In [6]:
import pandas as pd

train_texts = train_data["text"]
# train_texts[:5]

# Convert train_texts in Hugging face column to pandas series
train_texts = pd.Series(train_texts)
train_texts.head()

0    hey I am looking for Xray baggage datasets can...
1    "Get rich quick! Make millions in just days wi...
2    URGENT MESSAGE: YOU WON'T BELIEVE WHAT WE HAVE...
3    [Google AI Blog: Contributing Data to Deepfake...
4    Trying to see if anyone already has timestamps...
dtype: str

### **a) Removing Normal Duplicates: These are text that appear more than once**

#### **1) Count the number of duplicate emails**

In [7]:
duplicate_count = train_texts.duplicated().sum()
print("Number of duplicate emails:", duplicate_count)

Number of duplicate emails: 133


#### **2) Calculate the percentage of duplicate emails**

In [8]:
duplicate_percentage = (duplicate_count / len(train_texts)) * 100

print(f"Duplicate emails: {duplicate_count}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")

Duplicate emails: 133
Duplicate percentage: 1.63%


#### **3) Identify the duplicate emails**

In [9]:
# Get all rows containing duplicate emails
duplicate_emails = train_data.to_pandas()
# to_pandas() converts our Hugging Face dataset into a pandas DataFrame.
# duplicate_emails.head()

# duplicated(subset=["text"], keep=False) identifies every occurrence of duplicated email
# text, including the first occurrence.
duplicate_emails = duplicate_emails[
    duplicate_emails.duplicated(subset=["text"], keep=False)
]

# Display the first 10 duplicate entries
duplicate_emails.head(10)

,text,label
27,"Hello , I am looking for dataset for vibiratio...",not_spam
102,I've written a graphical CSV file viewer and e...,not_spam
169,"Hey, where can I find data to display most pop...",not_spam
181,I'm especially looking for their demographic d...,not_spam
183,"Hello , \n\nMe and my friends are writting a p...",not_spam
190,That data sets should contain the online image...,not_spam
231,"Show off, complain, and generally have a chat ...",not_spam
416,"Hello,\n\nI work as a Category Manager more a ...",not_spam
422,"Show off, complain, and generally have a chat ...",not_spam
452,Do anybody know where can I get the power cons...,not_spam


#### **4) Check for conflicting labels**

In [10]:
# Count the number of unique labels for each duplicated email
duplicate_emails.groupby("text")["label"].nunique().value_counts() 
# .nunique() method counts the number of distinct (unique) elements in a Pandas Series or 
# DataFrame

label
1    123
Name: count, dtype: int64

#### The result above means we have 123 duplicated email groups. Every duplicated email group has exactly `1` unique label.
#### There are no conflicting labels among the duplicates.

#### **5) Remove the duplicates emails**

In [11]:
train_data_cleaned = train_data.to_pandas().drop_duplicates(
    # subset=["text"]: Checks for duplicates based on email text.
    subset=["text"], 
    # keep="first": Keeps the first occurrence and removes subsequent copies.
    keep="first"
).reset_index(drop=True)
# reset_index(drop=True): Resets the row numbering after removal.

# Check the new dataset size
print("Original dataset size:", len(train_data))
print("Cleaned dataset size:", len(train_data_cleaned))
print("Duplicates remaining:", train_data_cleaned["text"].duplicated().sum())

Original dataset size: 8175
Cleaned dataset size: 8042
Duplicates remaining: 0


#### **6) Another duplicates we need to investigate are those in the test dataset**

##### **i) Check duplicates within the test data**

In [12]:
# Extract test email texts into pandas series
test_texts = pd.Series(test_data["text"])
test_texts[:5] 
# # OR
test_texts.head(5)

# Count the duplicate emails
test_duplicate_count = test_texts.duplicated().sum()
print("Number of duplicate emails in test data:", test_duplicate_count)

Number of duplicate emails in test data: 26


##### **ii) Check duplicates between training and test data**

In [13]:
# Find emails appearing in both training and test dataset
train_test_duplicates = train_data_cleaned[
    train_data_cleaned["text"].isin(test_texts)
]

print("Number of training emails also found in test data:",
      train_test_duplicates["text"].nunique())
# .nunique() method counts the number of distinct (unique) elements in a Pandas Series or 
# DataFrame

Number of training emails also found in test data: 79


##### **iii)  Check whether the 79 matching emails have consistent label**

##### **Why is this important?**
##### Imagine an email appears as:
##### - Training: spam
##### - Test: not_spam
##### That would indicate inconsistent labelling, even though the email text is identical.¶

In [14]:
# Convert test data to pandas DataFrame
test_data_df = test_data.to_pandas()
# print("--- Before Conversion ---")
# print(test_data[:5])

# print("\n--- After Conversion ---")
# If using a Jupyter Notebook, you can use 
# display(test_data_df.head(5))

# Keep one occurrence of each email in both datasets: i.e Remove exact duplicate 
# emails within each dataset individually, keeping only the first time that email appears.
train_unique = train_data_cleaned.drop_duplicates(subset=["text"]) # subset=["text"]: This tells 
# the function to only look at the "text" column when checking for duplicates. If two rows have 
# the exact same text, they are considered duplicates, even if their other columns (like an ID 
# or a label) are different.
# train_data_cleaned: This is your original DataFrame (likely a cleaned version of your training data).
# .drop_duplicates(...): This is the pandas function used to find and delete duplicate rows.

# test_unique: will contain a deduplicated version of your original test dataset, keeping only the
# first occurrence of each unique email text, Likewise train_unique
test_unique = test_data_df.drop_duplicates(subset=["text"])
# print(train_unique[:5])
# print(test_unique[:5])

# Match emails appearing in both datasets i.e finds identical text samples that exist in both 
# your training and testing datasets (a situation known as "data leakage"): So it creates a 
# side-by-side comparison table.
# overlapping_emails will contain only the rows where the exact same email text (i.e duplicates) 
# appears in both your training set and your testing set.
overlapping_emails = train_unique.merge(
    test_unique,
# train_unique.merge(test_unique, ...): This performs a database-style join between your 
# deduplicated training data and testing data.
    on="text",
# on="text": It matches rows where the content in the "text" column is exactly identical in both 
# datasets.
    how="inner",
# how="inner": This specifies an inner join, meaning the resulting DataFrame (overlapping_emails) 
# will only contain rows where the text exists in both the training and testing sets.
    suffixes=("_train", "_test")
# suffixes=("_train", "_test"): If your train and test sets have other columns with the same 
# name (like id or label), pandas will append these suffixes to differentiate them 
# (e.g., label_train and label_test).
)
# print(overlapping_emails.head())

# Check whether their labels agree
label_matches = (
    overlapping_emails["label_train"] == overlapping_emails["label_test"]
)
#  checks if the overlapping text samples have the exact same label in both the training and 
# testing datasets.

print("Total overlapping emails:", len(overlapping_emails))
print("Emails with matching labels:", label_matches.sum())
print("Emails with conflicting labels:", (~label_matches).sum())
# ~ : It flips all your boolean values—turning every True (matching labels) into False, and every 
# False (mismatched labels) into True.

Total overlapping emails: 79
Emails with matching labels: 79
Emails with conflicting labels: 0


##### **iv) Clean the test data**

In [15]:
# 1) Remove duplicate emails within the test
# cleans your testing dataset by removing duplicate rows based on text content and resetting the row 
# numbers.
test_data_cleaned = test_data_df.drop_duplicates(
# test_data_df.drop_duplicates(...): Looks through your raw test DataFrame to find identical text 
# entries.
    subset=["text"],
# subset=["text"]: Focuses only on the "text" column to determine what counts as a duplicate.
    keep="first"
# keep="first": Tells pandas to keep the very first time an email text appears and delete all 
# subsequent copies of it. (Note: This is the default behavior, but explicitly writing it makes the 
# code clearer).
).reset_index(drop=True)
# reset_index(drop=True): Re-indexes the rows starting from 0, 1, 2, .... Without this, your 
# DataFrame would have missing row numbers where the duplicates were deleted (e.g., jumping from row 
# 1 to row 4). drop=True ensures the old, broken index numbers are discarded completely.


# 2) Remove emails in the test data already present in the training data
# This code completely eliminates data leakage by removing any email from your test set that already
# exists in your training set.
test_data_cleaned = test_data_cleaned[
# test_data_cleaned["text"].isin(train_data_cleaned["text"]): This checks every email in your test 
# set and marks it True if it is also found in the training data, and False if it is brand new.
    ~test_data_cleaned["text"].isin(train_data_cleaned["text"])
# ~ (The Tilde Operator): This flips those values. It turns True (leaked data) into False, and False
# (unique test data) into True.
    
].reset_index(drop=True)


# 3)  Check the results
print("Original test dataset size:", len(test_data_df))
print("Cleaned test dataset size:", len(test_data_cleaned))
print("Duplicates remaining:", test_data_cleaned["text"].duplicated().sum())

# 4) # Check for overlap with training data
print("Remaining training-test overlap:",
      test_data_cleaned["text"].isin(train_data_cleaned["text"]).sum())

Original test dataset size: 2725
Cleaned test dataset size: 2620
Duplicates remaining: 0
Remaining training-test overlap: 0


## **b) Removing near-duplicates for `training data`**

#### Across training and test data: Near-identical emails could still cause data leakage, even after removing exact duplicates.
#### We'll use a technique called Sequence Matching initially. Python's difflib module can compare two text strings and calculate a similarity ratio between 0 and 1.
#### For example:
#### - 1.0 means the texts are identical.
#### - 0.9 means they are very similar.
#### - 0.5 means they have moderate similarity.
#### - 0.0 means they have no matching sequence.

#### For our dataset of 8,175 training emails, I recommend using TF-IDF with character n-grams instead of comparing every possible pair using difflib.
#### **Why?** Comparing every pair directly can be computationally expensive. TF-IDF with character n-grams is more practical for finding similar email text, including small spelling and punctuation differences.

#### **i) Prepare the text for similarity checking**
##### First, let's import the tools we'll need.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
# TfidfVectorizer: Converts email text into numerical representations based on character
# patterns.
from sklearn.neighbors import NearestNeighbors
# NearestNeighbors: Finds emails whose numerical representations are most similar.

#### **ii) Convert training emails into TF-IDF character n-grams**
#### **Why are we doing this?**
##### We want to identify emails that are similar even when they have small differences in spelling, punctuation or formatting.

In [17]:
# Create the TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    analyzer="char", 
    # analyzer="char": Examines individual characters rather than whole words.
    ngram_range=(3, 5),
    # ngram_range=(3, 5): Extracts character sequences of 3, 4 and 5 characters.
    min_df=2
    # min_df=2: Ignores character patterns appearing in fewer than two emails.
)

# Convert training emails into numerical representations
train_tfidf = tfidf_vectorizer.fit_transform(
# fit_transform(): Learns the character patterns and converts the emails into numerical representations.
    train_data_cleaned["text"]
)

# Check the shape
print("TF-IDF matrix shape:", train_tfidf.shape)

TF-IDF matrix shape: (8042, 171459)


#### **Let's understand these numbers:**
#### - 8,042 rows: We have 8,042 unique training emails after removing exact duplicates.
#### - 171,459 columns: The vectorizer identified 171,459 different character patterns (n-grams) across the emails.
#### Each row represents an email, and each column represents a character pattern.

#### ***iii) Find similar emails***
##### Now we'll use NearestNeighbors to find emails that are similar to one another.

In [18]:
# Create the nearest neighbors model
nearest_neighbors = NearestNeighbors(
    n_neighbors=3,
# n_neighbors=3: Finds the three closest emails for each email, including the email itself.
    metric="cosine",
# metric="cosine": Measures how similar two email representations are.
    algorithm="brute"
# algorithm="brute": Compares against all other emails to find the nearest neighbors.
)

# Fit the model on the TF-IDF matrix
nearest_neighbors.fit(train_tfidf)
# fit(): Prepares the model using our TF-IDF representations.

# Find the nearest neighbors for each email
distances, indices = nearest_neighbors.kneighbors(train_tfidf)
# kneighbors(): Finds the nearest emails and returns their distances and row indices.

# Check the shapes
print("Distances shape:", distances.shape)
print("Indices shape:", indices.shape)

Distances shape: (8042, 3)
Indices shape: (8042, 3)


##### **This means:**
##### - We have 8,042 training emails.
##### - For each email, the algorithm found 3 nearest neighbours.
##### - distances contains the cosine distances.
##### - indices contains the positions of the corresponding emails.
##### Remember, the first neighbour is normally the email itself, with a distance of 0.

#### ***iv) Inspect the most similar emails***
##### Now let's identify emails that have at least 90% similarity, excluding each email's match with itself.

In [19]:
# Set our similarity threshold
similarity_threshold = 0.90
# similarity_threshold = 0.90: We are looking for emails with at least 90% similarity.

# Store potential near-duplicate pairs
near_duplicates = []

# Loop through every email
for i in range(len(train_data_cleaned)):
    
    # Loop through its nearest neighbors
    for j in range(1, distances.shape[1]):
# range(1, distances.shape[1]): Skips the first neighbour, which is normally the email itself.

        # Calculate similarity
        similarity = 1 - distances[i, j]
# # 1 - distances[i, j]: Converts cosine distance into similarity.

        # Check if similarity meets our threshold
        if similarity >= similarity_threshold:

            # Store the email pair and similarity score
            near_duplicates.append((
# near_duplicates.append(): Stores the positions of the two emails and their similarity score.
                i,
                indices[i, j],
                similarity
            ))

# Display the number of potential near-duplicate pairs
print("Potential near-duplicate pairs:", len(near_duplicates))

Potential near-duplicate pairs: 24


##### Good! We've found 24 potential near-duplicate pairs with at least 90% similarity.

##### However, before interpreting this number, there are two things to understand:
##### 1. The same pair might appear twice because email A can identify email B as a neighbour, and email B can identify email A.
##### 2. We asked for only 3 nearest neighbours per email, including itself. Therefore, we are checking only the two closest other emails for each email. We may be missing some additional matches.
##### Let's first inspect the 24 pairs we found.

#### ***v) Remove repeated pair entries***
##### The 24 matches may include the same pair twice because email A can identify email B, and email B can also identify email A. Let's remove those repeated entries.

In [20]:
# Convert potential near-duplicates into a DataFrame
# It takes a raw list/iterable (near_duplicates) and turns it into a clean pandas DataFrame.
# It assigns three specific columns: the first email, the second email, and their calculated 
# similarity score.
near_duplicates_df = pd.DataFrame(
    near_duplicates,
    columns=["email_1", "email_2", "similarity"]
)

# Remove duplicate pairs, regardless of their order
# This creates a  temporary column called pair.
near_duplicates_df["pair"] = near_duplicates_df.apply(
    lambda row: tuple(sorted([row["email_1"], row["email_2"]])),
# If the data contains both (alice@email.com, bob@email.com) and (bob@email.com, alice@email.com), 
# sorting the names alphabetically ensures both rows result in the exact same 
# tuple: ('alice@email.com', 'bob@email.com').

    axis=1
)


# This looks at the new pair column and throws away any rows that are identical.
near_duplicates_df = near_duplicates_df.drop_duplicates(
    subset="pair"
).drop(columns="pair")
# Once the duplicate pairs are removed, it deletes the temporary pair column to keep the 
# DataFrame clean.



# Sort by similarity, highest first
# It sorts the entire list so that the most identical emails appear at the very top 
# (ascending=False).
near_duplicates_df = near_duplicates_df.sort_values(
    by="similarity",
    ascending=False
).reset_index(drop=True)
#  It resets the row numbers (index) so they go sequentially from 0 to the end, instead of 
# keeping the jumbled, post-sorted index numbers.

print("Unique near-duplicate pairs:", len(near_duplicates_df))

near_duplicates_df

Unique near-duplicate pairs: 14


,email_1,email_2,similarity
0,3296,4137,0.999136
1,890,4137,0.995774
2,890,3296,0.994914
3,4195,4137,0.979448
4,4195,3296,0.978607
5,6030,7214,0.932412
6,3282,5151,0.925640
7,244,772,0.924116
8,244,7029,0.923555
9,5514,244,0.920758


### **The steps that was followed in `going-thru-some-parts-of-email-spam-project.ipynb` is bcos the emails that are duplicated are few i.e 14. But in a situation whereby the emails are tens of thousands, we can not follow that procedure but this below bcos u can't inspect them:**

#### ***vi) Automatically remove near-duplicates***

In [21]:
# 1. Catch 100% exact matches across the entire dataset instantly
# Replaces checking `email_1 == email_2` one by one
train_data_cleaned = train_data_cleaned.drop_duplicates(subset=["text"]).reset_index(drop=True)
# drop_duplicates(subset=["text"]) eliminates all identical pairs immediately. This removes 
# the need to manually check if two texts are 100% equal.

# 2. Set an automatic similarity cutoff (e.g., 0.95)
# Replaces your manual `near_duplicate_review` table and human reading
THRESHOLD = 0.95
# Setting a THRESHOLD (such as 0.95) acts as your automated decision-maker. Anything above this 
# score is treated as a true near-duplicate, while anything below it is kept without human review.
auto_drop_pairs = near_duplicates_df[near_duplicates_df["similarity"] >= THRESHOLD]

# 3. Collect all duplicate indices into a single list and drop them in bulk
# Replaces hardcoding single indices like `index=5637`
indices_to_drop = auto_drop_pairs["email_2"].astype(int).unique()
train_data_cleaned = train_data_cleaned.drop(index=indices_to_drop).reset_index(drop=True)

print(f"Automatically removed {len(indices_to_drop)} near-duplicate emails.")
print("New training dataset shape:", train_data_cleaned.shape)

Automatically removed 2 near-duplicate emails.
New training dataset shape: (8040, 2)


#### **One Optional Safety Check**
##### If you want to ensure you aren't deleting emails that have conflicting labels (like one marked "Spam" and the other marked "Ham"), add a quick filter before Step 3:

In [22]:
# Map labels to verify they match before dropping
label_1 = auto_drop_pairs["email_1"].apply(lambda x: train_data_cleaned.iloc[int(x)]["label"])
label_2 = auto_drop_pairs["email_2"].apply(lambda x: train_data_cleaned.iloc[int(x)]["label"])

# Only drop pairs where both emails share the exact same label
safe_to_drop = auto_drop_pairs[label_1 == label_2]
indices_to_drop = safe_to_drop["email_2"].astype(int).unique()

# Print the results so you can see what happened
print("Total pairs above threshold:", len(auto_drop_pairs))
print("Pairs with matching labels (safe to drop):", len(safe_to_drop))
print("Unique row indices ready to drop:", indices_to_drop)

Total pairs above threshold: 5
Pairs with matching labels (safe to drop): 3
Unique row indices ready to drop: [4137 3296]


#### **If you have 10,000 near-duplicate pairs, you don't need to look at the ones where both emails have the same label (e.g., both are "Spam"). You only need to catch data corruption—where two identical emails have completely different labels.**
#### **This code filters down thousands of rows to only show you pairs with conflicting labels:**

In [23]:
# # 1. Map labels directly onto your pair DataFrame
# near_duplicates_df["label_1"] = near_duplicates_df["email_1"].apply(lambda idx: train_data_cleaned.iloc[int(idx)]["label"])
# near_duplicates_df["label_2"] = near_duplicates_df["email_2"].apply(lambda idx: train_data_cleaned.iloc[int(idx)]["label"])

# # 2. Filter for rows where the labels do NOT match
# conflicting_pairs = near_duplicates_df[near_duplicates_df["label_1"] != near_duplicates_df["label_2"]]

# print(f"Total near-duplicates: {len(near_duplicates_df)}")
# print(f"Pairs with conflicting labels needing manual review: {len(conflicting_pairs)}")

### **NOTE:**
### **`0.90` is a great threshold for checking `train-to-test data leakage`, while `0.95` is safer when `automatically deleting duplicates inside your training data`.**

## **c) Removing near-duplicates for `test data`**
#### ***1) Convert test emails into TF-IDF features***
#### We'll reuse the tfidf_vectorizer we fitted on the training data. This is important because we want both datasets represented using the same vocabulary.

In [24]:
test_tfidf = tfidf_vectorizer.transform(
    # transform() converts the test emails into TF-IDF features.
    # We use transform(), not fit_transform(), because the vectorizer has already learned 
    # its character n-gram vocabulary from the training data.
    test_data_cleaned["text"]
)

print("TF-IDF matrix shape:", test_tfidf.shape)

TF-IDF matrix shape: (2620, 171459)


#### ***2) Find the nearest neighbours***
#### Now we'll find the most similar emails within the test dataset.

In [25]:
nearest_neighbors_test = NearestNeighbors(
    n_neighbors=6, # n_neighbors=6 finds six nearest neighbours for each email, including itself.
    metric="cosine", # metric="cosine" measures how similar the emails are.
    algorithm="brute"
)

nearest_neighbors_test.fit(test_tfidf)

# test_distances stores the distances between emails.
test_distances, test_indices = nearest_neighbors_test.kneighbors(test_tfidf)
# test_indices stores the positions of their nearest neighbours.

print("Distances shape:", test_distances.shape)
print("Indices shape:", test_indices.shape)

Distances shape: (2620, 6)
Indices shape: (2620, 6)


#### ***Step 3: Identify potential near-duplicates***
#### We'll use the same similarity threshold of 0.90 that we used for the training dataset.

In [26]:
similarity_threshold = 0.90
test_near_duplicates = []

for i in range(len(test_data_cleaned)):
    for j in range(1, test_distances.shape[1]): 
        # We skip j = 0 because that is the email itself.
        similarity = 1 - test_distances[i, j]
        # 1 - distance converts cosine distance into cosine similarity.

        if similarity >= similarity_threshold:
            test_near_duplicates.append((
                i,
                test_indices[i, j],
                similarity
            ))

print("Potential near-duplicate pairs:", len(test_near_duplicates))

Potential near-duplicate pairs: 0


### **NOTE FOR ABOVE `Removing near-duplicates for test data`:**
### **When i asked google it told me that in test data, there is no need for removing near duplicates bcos it hides mistakes. It said it is not adviceable.** 
### **For e.g If a scammer sends the same slightly edited email 10 times in real life, your model will face that email 10 times.**
### **But it said cleaning test data is essential.**

## **d) Removing near-duplicates between the training and test datasets.**
#### This is important because similar emails appearing in both datasets could affect our model evaluation.

#### ***1) Find the nearest training emails for each test email***

In [27]:
cross_neighbors = NearestNeighbors(
    n_neighbors=6,
    metric="cosine",
    algorithm="brute"
)

cross_neighbors.fit(train_tfidf)
# fit(train_tfidf) stores the training emails as our reference dataset.

# kneighbors(test_tfidf) searches for the six most similar training emails for every test 
# email.
cross_distances, cross_indices = cross_neighbors.kneighbors(test_tfidf)
# cross_distances stores the similarity distances.
# cross_indices stores the positions of the matching training emails.

print("Distances shape:", cross_distances.shape)
print("Indices shape:", cross_indices.shape)

Distances shape: (2620, 6)
Indices shape: (2620, 6)


#### ***2) Identify potential near-duplicates across both datasets***
#### We'll use the same similarity threshold of 0.90. Unlike our earlier checks within a single dataset, we will examine all six neighbours because none of them is the test email itself.

In [28]:
similarity_threshold = 0.90
cross_near_duplicates = []

for i in range(len(test_data_cleaned)): # i: position of the email in the test dataset.
    for j in range(cross_distances.shape[1]):
        similarity = 1 - cross_distances[i, j]
        
        if similarity >= similarity_threshold:
            cross_near_duplicates.append((
                i,
                cross_indices[i, j],
            # cross_indices[i, j]: position of its matching email in the training dataset.
                similarity
                # similarity: how similar the two emails are.
            ))

print("Potential near-duplicate pairs:", len(cross_near_duplicates))

Potential near-duplicate pairs: 2


#### Good! We found 2 potential near-duplicate pairs between the training and test datasets.
#### This means two test emails have at least one training email with a similarity score of 0.90 or higher

#### ***3) Display the two candidate pairs***
#### Let's create a DataFrame to make the results easier to inspect.

In [29]:
cross_near_duplicates_df = pd.DataFrame(
    cross_near_duplicates,
    columns=["test_email", "train_email", "similarity"]
    # test_email: position of the email in the test dataset.
    # train_email: position of the matching email in the training dataset.
    # similarity: similarity score between the two emails.
)

cross_near_duplicates_df = cross_near_duplicates_df.sort_values(
    by="similarity",
    ascending=False
).reset_index(drop=True)

cross_near_duplicates_df

,test_email,train_email,similarity
0,2100,7910,0.968842
1,1442,244,0.947020


#### ***4) Skipped in large datasets / Manual review in small datasets***
#### This is bcos in future no one will be reviewing small datasets but rathere larger datasets which this number 4 process will not work. Imagine review just 50 near-duplicates emials with ur eyes not to talk of 100 or 500 or 20000.

#### ***5) Remove leaked near-duplicates from test data.***
#### we are removing it strictly from the test data, specifically to prevent data leakage.
#### You never delete these overlapping rows from your training data because you want your model to keep as many examples as possible to learn from.

In [30]:
# Collect the unique test indices that overlap with training data
test_indices_to_drop = cross_near_duplicates_df["test_email"].astype(int).unique()

# Remove them from the test set only
test_data_cleaned = test_data_cleaned.drop(index=test_indices_to_drop).reset_index(drop=True)

print("Updated test dataset shape:", test_data_cleaned.shape)

Updated test dataset shape: (2618, 2)


### **NOTE: One Quick Pro-Tip for Real-World Scale**
### **When your dataset eventually grows past 50,000+ emails, remember one tiny tweak in NearestNeighbors:**
### **—Change `algorithm="brute"` to `algorithm="auto"` `(or use a dedicated vector search library like FAISS)`, because brute-force comparison slows down significantly once you cross hundreds of thousands of rows.**

## **B) Text cleaning and normalization**
#### **The next step is checking for missing or empty email messages before we begin text cleaning and normalization.**
#### **We want to know whether any email contains:**
#### - A missing (NaN) text value.
#### - An empty string ("").
#### - Only whitespace (spaces, tabs, or newlines).
#### **Let's start with the training dataset.**

#### ***1) Check for missing email text***
#### Below is a fc that checks for missing email text

In [31]:
def check_email_texts(data, dataset_name):
    """
    Checks for missing, empty, and whitespace-only email messages.
    """
    # missing: counts missing text values (NaN).
    missing = data["text"].isna().sum()

    # fillna(""): temporarily replaces missing values with empty strings for checking.
    # astype(str): ensures non-string types (like numbers) don't crash .str.strip().
    empty = (
        # str.strip(): removes surrounding whitespace for the purpose of the check
        data["text"].fillna("").astype(str).str.strip() == ""
    ).sum()
    # == "": identifies empty or whitespace-only messages.
    
    print(f"{dataset_name} dataset:")
    print("Missing emails:", missing)
    print("Empty or whitespace-only emails:", empty)

#### **USe the fc**

In [32]:
check_email_texts(train_data_cleaned, "Training")
check_email_texts(test_data_cleaned, "Test")

Training dataset:
Missing emails: 0
Empty or whitespace-only emails: 0
Test dataset:
Missing emails: 0
Empty or whitespace-only emails: 0


#### **Both datasets have zero missing emails and zero empty or whitespace-only messages.We don't need to remove anything at this stage.**

#### ***2) Inspect the email text before cleaning***
#### **Before writing a cleaning function, we need to understand what appears in the emails. We should inspect:**
#### 1. HTML tags.
#### 2. URLs and links.
#### 3. Email addresses.
#### 4. Special characters and punctuation.
#### 5. Numbers and currency symbols.
#### We shouldn't automatically remove these because some of them may contain useful spam indicators.
#### **Let's create a reusable function to display sample emails.**
#### **i) Create the function**
##### Include this step. Even when working with millions of rows, inspecting a fixed, small number of samples (n=5) is an essential sanity check to catch weird formatting (like HTML entities or links) before writing cleaning rules.

In [33]:
def display_email_samples(data, dataset_name, n=5):
    """
    Displays sample email messages for inspection.
    """
    print(f"{dataset_name} email samples:\n")

    for i in range(n):
        print(f"Email {i + 1}:")
        print(data["text"].iloc[i])
        print("-" * 80)

#### **ii) Inspect both datasets**

In [34]:
display_email_samples(train_data_cleaned, "Training")
display_email_samples(test_data_cleaned, "Test")

Training email samples:

Email 1:
hey I am looking for Xray baggage datasets can you provide me with the same 
--------------------------------------------------------------------------------
Email 2:
"Get rich quick! Make millions in just days with our new and revolutionary system! Don't miss out on this amazing opportunity!"


--------------------------------------------------------------------------------
Email 3:
URGENT MESSAGE: YOU WON'T BELIEVE WHAT WE HAVE TO OFFER!!!

Hey you! Yeah, you with the eyes reading this right now. Do you want to be the coolest cat on the block? Do you want to get all the likes, hearts and emojis? Do you want to be ~*POPULAR*~? Well, we have the solution for you.

Introducing our brand new feature that will blow your mind and your feed! We can't give away too many details, but let's just say it involves puppies, unicorns, and a drum kit.

But wait, there's more! Sign up
--------------------------------------------------------------------------------
Em

#### **One important point: legitimate emails can also contain URLs, HTML and special characters. We shouldn't assume that these features automatically indicate spam.**

#### **iii) Building our reusable text-cleaning function.**
#### So in **`going-thru-some-parts-of-email-spam-project.ipynb`** this i did not repeat the steps done from **`a) to g)`**. This is bcos it was being explained so oit was done in difft steps. But now we can just do the whole in one step which is the right way and is efficient way to go when performing the steps **`a) to g)`**.

#### **a) Create a fc that will do the below cleaning**
#### **How Your Tutor's Steps Map to That Single Function**
##### Sub-step a (import html): Became html.unescape(str(text)) at the start of the function.
##### Sub-step b (Markdown links): Became MD_LINK_RE.sub(r"\1", text) (plus BLOCKQUOTE_RE to strip > quote symbols).
##### Sub-step c (Standalone URLs): Became URL_RE.sub(" URL ", text).
##### Sub-step d (Email addresses): Became EMAIL_RE.sub(" EMAIL ", text).
##### Sub-step e (Lowercase & whitespace): Became text = text.lower() and WHITESPACE_RE.sub(" ", text).strip().
##### Sub-step f (Escaped backslashes): Became text = text.replace("\\", "").

In [35]:
import html
import re

# Compile regex patterns once for fast execution on large datasets
BLOCKQUOTE_RE = re.compile(r"(?m)^\s*>\s?")
MD_LINK_RE = re.compile(r"\[([^\]]+)\]\([^)]+\)")
URL_RE = re.compile(r"https?://\S+")
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b")
WHITESPACE_RE = re.compile(r"\s+")

def clean_email(text):
    """
    Cleans and normalizes email text efficiently for large datasets.
    """
    # Ensure input is a string in case of unexpected data types
    text = html.unescape(str(text))

    # Remove Markdown blockquote markers and link formatting
    text = BLOCKQUOTE_RE.sub("", text)
    text = MD_LINK_RE.sub(r"\1", text)

    # Lowercase first so special tokens stay uppercase
    text = text.lower()

    # Replace URLs and email addresses with distinct uppercase tokens
    text = URL_RE.sub(" URL ", text)
    text = EMAIL_RE.sub(" EMAIL ", text)

    # Remove escape backslashes and normalize whitespace
    text = text.replace("\\", "")
    text = WHITESPACE_RE.sub(" ", text).strip()

    return text


#### **b) Apply the function to both datasets**
#### We're creating a new column called clean_text rather than overwriting text. This allows us to compare the original emails with their cleaned versions.

In [36]:
# Clean training email texts
train_data_cleaned["clean_text"] = train_data_cleaned["text"].apply(clean_email)

# Clean test email texts
test_data_cleaned["clean_text"] = test_data_cleaned["text"].apply(clean_email)

#### **c) Compare original and cleaned emails**
#### Let's inspect a few examples to ensure the cleaning function worked correctly on real dataset messages, not just our sample.

In [37]:
def compare_cleaned_emails(data, dataset_name, n=5):
    """
    Displays original and cleaned email texts side by side.
    """

    print(f"{dataset_name} dataset:\n")

    for i in range(n):
        print(f"Email {i + 1}")
        print("Original:")
        print(data["text"].iloc[i])

        print("\nCleaned:")
        print(data["clean_text"].iloc[i])

        print("-" * 80)

#### **Use it:**

In [38]:
compare_cleaned_emails(train_data_cleaned, "Training")
compare_cleaned_emails(test_data_cleaned, "Test")

Training dataset:

Email 1
Original:
hey I am looking for Xray baggage datasets can you provide me with the same 

Cleaned:
hey i am looking for xray baggage datasets can you provide me with the same
--------------------------------------------------------------------------------
Email 2
Original:
"Get rich quick! Make millions in just days with our new and revolutionary system! Don't miss out on this amazing opportunity!"



Cleaned:
"get rich quick! make millions in just days with our new and revolutionary system! don't miss out on this amazing opportunity!"
--------------------------------------------------------------------------------
Email 3
Original:
URGENT MESSAGE: YOU WON'T BELIEVE WHAT WE HAVE TO OFFER!!!

Hey you! Yeah, you with the eyes reading this right now. Do you want to be the coolest cat on the block? Do you want to get all the likes, hearts and emojis? Do you want to be ~*POPULAR*~? Well, we have the solution for you.

Introducing our brand new feature that will blow

#### **Overall, the cleaning function is working well, and we can move towards tokenization.**

# **Stage 3: Preparing Data for Model Training**
#### **C) Splitting training and validation data.**

#### **D) Tokenization**
#### Break each email into smaller units called tokens (words or subwords).

#### **E) Vocabulary creation**
#### Build a collection of tokens the model will recognize.

#### **F) Numerical conversion**
#### Assign integer IDs to tokens so the neural network can process them.

#### **G) Padding and truncation**
#### Make sequences the same length for batch processing

#### **H) Label encoding.**

## **Step C: Create a function for splitting the data**

In [39]:
from sklearn.model_selection import train_test_split

def split_data(data, test_size=0.2, random_state=42):
    """
    Splits email data into training and validation sets.
    """

    # Separate features and labels
    X = data["clean_text"]
    y = data["label"]

    # Split the data
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    return X_train, X_valid, y_train, y_valid

### **Use the fc**

In [40]:
X_train, X_valid, y_train, y_valid = split_data(train_data_cleaned)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_valid))
print("\nTraining label distribution:\n", y_train.value_counts(normalize=True))
print("\nValidation label distribution:\n", y_valid.value_counts(normalize=True))

Training samples: 6432
Validation samples: 1608

Training label distribution:
 label
spam        0.51306
not_spam    0.48694
Name: proportion, dtype: float64

Validation label distribution:
 label
spam        0.51306
not_spam    0.48694
Name: proportion, dtype: float64


## **D) Tokenization**
#### **TensorFlow's TextVectorization layer** is what we will use for tokenization and converting our email text into numerical sequences.
#### **It performs several related tasks:**
#### 1. Tokenization: Splits each email into individual tokens (usually words).
#### 2. Vocabulary creation: Builds a collection of unique tokens from the training emails.
#### 3. Integer encoding: Assigns each token a numerical ID.
#### For example:
#### **Original email:**
#### "Get your prize now"

#### **Tokenized:**
#### ["get", "your", "prize", "now"]

#### **Integer encoded:**
#### [15, 8, 42, 6]

#### ***1) Create the TextVectorization layer***
#### We set max_tokens=10000 as an upper limit, not a requirement to use exactly 10,000 tokens.

In [41]:
# Create the layer
text_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=10000, # max_tokens=10000: Limits the vocabulary to a maximum of 10,000 tokens
    # of most frequent words prevents rare typos, random hashes, or spammer gibberish from 
    # bloating your model's embedding matrix..
    output_mode="int", # output_mode="int": Converts tokens into integer IDs.
    standardize=None
    # standardize=None: Preserves our existing text formatting, including punctuation, rather 
    # than automatically stripping punctuation.
)

#### ***Step 2: Adapt the TextVectorization layer***
#### **What does .adapt() do?**
#### **It examines the training emails and:**
#### 1. Identifies the individual tokens (words).
#### 2. Builds a vocabulary of the tokens it finds.
#### 3. Assigns each token an integer ID.
#### 4. Keeps the most frequent tokens, subject to our max_tokens=10000 limit.
#### **For example, if the word `free` appears frequently, it will receive an `integer ID` in the vocabulary.**

In [ ]:
text_vectorizer.adapt(X_train)
# Passing only X_train into .adapt() strictly prevents data leakage from your validation or
# test sets.

## **E) Vocabulary creation**
#### ***1) Retrieve the vocabulary***
#### I used only this step here bcos when working with large datasets 2, 3, 4, 5 as i did in **`going-thru-some-parts-of-email-spam-project.ipynb`**

In [ ]:
vocabulary = text_vectorizer.get_vocabulary()

print("Vocabulary size:", len(vocabulary))
print("\nFirst 20 tokens:")
print(vocabulary[:20])

#### **What are we doing?**
#### - get_vocabulary() retrieves the words and tokens learned by the vectorizer.
#### - len(vocabulary) tells us how many tokens are in the vocabulary.
#### - vocabulary[:20] displays the first 20 tokens.
#### **One important detail: the first two entries are reserved for special purposes:**
#### - "" — padding.
#### - "[UNK]" — tokens that are not in the vocabulary (unknown words).
#### **The remaining tokens are arranged by frequency, from most frequent to less frequent.**

## **F) Numerical conversion (Setting up the Fixed-Length Vectorizer)**
#### When dealing with a big dataset, these are the main steps to follow

#### ***1) Splits each training email into words and counts how many words are in each email.***

In [ ]:
train_lengths = X_train.str.split().str.len()

# Prints the maximum, average, and 95th percentile word counts so we can pick a smart sequence length.
print("Longest training email:", train_lengths.max())
print("Average training email length:", round(train_lengths.mean(), 2))
print("95th percentile:", train_lengths.quantile(0.95))

#### ***2) Defines a reusable function to build and train our text-to-number converter.***
#### **Why 150?**
#### - It captures more than 95% of the typical email lengths.
#### - It reduces unnecessary padding.
#### - It keeps the input sequences smaller, which is useful when training on your laptop's CPU.
#### - A small number of longer emails will be truncated, but we can evaluate whether that affects performance later.

In [ ]:
def create_text_vectorizer(train_texts, max_tokens=10000, sequence_length=150):
    """
    Creates and adapts a fixed-length TextVectorization layer.
    """
    # Initializes TensorFlow's TextVectorization layer.
    vectorizer = tf.keras.layers.TextVectorization(
        max_tokens=max_tokens,# Caps the vocabulary at the 10,000 most frequent tokens.
        output_mode="int", # Outputs whole numbers (integer IDs) instead of raw text.
        output_sequence_length=sequence_length, 
        # Pads short emails with 0s and cuts long emails at 150 tokens.
        standardize=None # Skips built-in text cleaning since we already cleaned the text.
    )
    
    # Scans ONLY the training emails to build the vocabulary and assign integer IDs.
    vectorizer.adapt(train_texts)
    
    # Returns the ready-to-use vectorizer.
    return vectorizer

#### ***3) Builds our vectorizer using X_train, capping vocabulary at 10,000 and sequence length at 150.***

In [ ]:
text_vectorizer = create_text_vectorizer(X_train, max_tokens=10000, sequence_length=150)

# Sanity check: Converts just the very first training email to verify words become numbers properly.
print("\nOriginal sample email:\n", X_train.iloc[0])
print("\nVectorized sample email:\n", text_vectorizer(X_train.iloc[:1])[0].numpy())

## **G) Padding and Truncation (Transforming All Datasets to Uniform Shapes)**

#### ***1) Converting all trainings and validation emails to uniform matrix***

In [ ]:
# Converts all training emails into a uniform matrix of shape (num_train_emails, 150).
X_train_vectorized = text_vectorizer(X_train)

# Converts all validation emails using the exact same rules and vocabulary.
X_valid_vectorized = text_vectorizer(X_valid)

# Prints the final dimensions of all three datasets to confirm every row has length 150.
print("\nTraining shape:", X_train_vectorized.shape)
print("Validation shape:", X_valid_vectorized.shape)

#### ***2) Converting test emails to uniform matrix***

In [ ]:
# Extracts the cleaned text and labels from our final test dataset.
X_test = test_data_cleaned["clean_text"]
y_test = test_data_cleaned["label"]

# Converts all test emails into a uniform matrix of shape (num_test_emails, 150).
X_test_vectorized = text_vectorizer(X_test)

# Prints the final dimensions of all three datasets to confirm every row has length 150.
print("Test shape:", X_test_vectorized.shape)

## **H) Preparing our labels OR Label encoding.**

#### ***1) Inspect class balance across all three splits (fast even on millions of rows).***

In [ ]:
print("Training label distribution:\n", y_train.value_counts())
print("\nValidation label distribution:\n", y_valid.value_counts())
print("\nTest label distribution:\n", y_test.value_counts())

#### Good. Both classes are represented in all three datasets, and their distributions are reasonably balanced.

#### Let's interpret the training data:
#### - Spam: 3,299 emails.
#### - Not spam: 3,133 emails.
#### **This is approximately a 51.3% / 48.7% distribution. We don't have a major class imbalance problem.**

#### ***2) Define a reusable function to convert text labels into binary integers (0 and 1).***
#### **We'll convert:**
#### - not_spam → 0
#### - spam → 1
#### **Why use these values?** They give us a binary classification target that our neural network can learn to predict.

In [ ]:
def encode_labels(labels):
    """
    Converts 'not_spam' to 0 and 'spam' to 1, and ensures no unmapped NaN values exist.
    """
    # Maps text categories to integers.
    encoded = labels.map({"not_spam": 0, "spam": 1})
    
    # Safety check for large datasets: stops immediately if any unexpected label turned into NaN.
    assert encoded.isna().sum() == 0, "Error: Found unexpected label values that became NaN!"
    
    # Returns the clean integer Series.
    return encoded.astype("int32")

#### ***3) Apply the encoding function to training, validation, and test labels.***
#### **The below converts the text labels into numerical values:**
#### - not_spam → 0
#### - spam → 1
#### We need numerical labels because the neural network will learn to distinguish between the two classes using these values.

In [ ]:
y_train_encoded = encode_labels(y_train)
y_valid_encoded = encode_labels(y_valid)
y_test_encoded = encode_labels(y_test)

#### ***4) Convert pandas Series into fast NumPy arrays for TensorFlow model training.***

In [ ]:
y_train_encoded = y_train_encoded.to_numpy()
y_valid_encoded = y_valid_encoded.to_numpy()
y_test_encoded = y_test_encoded.to_numpy()

# Verify final shapes match between email matrices (X) and target labels (y).
print("\nFinal Shapes Verification:")
print(f"Training:   X={X_train_vectorized.shape}, y={y_train_encoded.shape}, dtype={y_train_encoded.dtype}")
print(f"Validation: X={X_valid_vectorized.shape}, y={y_valid_encoded.shape}, dtype={y_valid_encoded.dtype}")
print(f"Test:       X={X_test_vectorized.shape}, y={y_test_encoded.shape}, dtype={y_test_encoded.dtype}")

# **Building the Email Spam Identification Model**
#### We are now moving from data preparation to building our deep-learning model.
#### We'll start with an **`Embedding layer followed by a GlobalAveragePooling1D layer and Dense layers.`** This is a suitable baseline for our email text-classification task.

### ***Step 1: Understand our model architecture***
#### **1. Input layer**
#### Receives each email as a sequence of 150 integers from our TextVectorization layer.

#### **2. Embedding layer**
#### Converts each integer token into a trainable vector of numbers. Instead of treating words as unrelated numerical IDs, the model learns representations that help it recognise useful patterns.

#### **3. GlobalAveragePooling1D**
#### Takes the average of the word vectors across the sequence, producing one summary vector for each email.

#### **4. Dense layer**
#### Learns combinations of patterns that help distinguish spam from legitimate emails.

#### **5. Output layer**
#### Produces one probability between 0 and 1:
#### - Closer to 1: likely spam.
#### - Closer to 0: likely not spam.

### ***Step 2: Understand the Embedding layer parameters***
#### We will use two important parameters:

#### tf.keras.layers.Embedding(
####    input_dim=10000,
####    output_dim=128
#### )

#### **`input_dim=10000: maximum vocabulary size available to the model`.**
#### **`output_dim=128: each token will be represented by a vector of 128 numbers.`**

## **Build, Summarize, and Compile the Email Classification Model**
#### This is the code for the above part

#### ***1) Define the model architecture***

In [ ]:
email_model = tf.keras.Sequential([
    # Embedding: Turns each of the 10,000 word IDs into a dense 128-number vector.
    # mask_zero=True tells later layers to ignore padding zeros (0) when averaging.
    tf.keras.layers.Embedding(
        input_dim=10000,
        output_dim=128,
        mask_zero=True
    ),

    # GlobalAveragePooling1D: Averages the word vectors across the email into one summary vector.
    tf.keras.layers.GlobalAveragePooling1D(),

    # Dense(64): Hidden layer with 64 neurons to learn combinations of spam patterns.
    tf.keras.layers.Dense(64, activation="relu"),

    # Dense(1): Output layer producing a single probability between 0 (not_spam) and 1 (spam).
    tf.keras.layers.Dense(1, activation="sigmoid")
])

#### ***2) Display the initial model summary***
#### This helps us understand the model's structure and how many trainable parameters it contains before training

In [ ]:
email_model.summary()

#### ***3) Build the model and inspect the summary***

In [ ]:
# Explicitly sets the input shape to (batch_size, 150 tokens) and prints layer parameters.
email_model.build(input_shape=(None, 150))
email_model.summary()

#### ***4) Compile the model***

In [ ]:
# Attaches the loss function, optimizer, and evaluation metrics to prepare for training.
email_model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),      # Standard loss for binary (0 vs 1) classification.
    optimizer=tf.keras.optimizers.Adam(),           # Adaptive optimizer that updates weights efficiently.
    metrics=[
        "accuracy",                                 # Overall percentage of correct predictions.
        tf.keras.metrics.Precision(name="precision"), # Out of emails flagged as spam, how many actually were?
        tf.keras.metrics.Recall(name="recall")        # Out of all actual spam emails, how many did we catch?
    ]
)

# **Training the Email Spam Identification Model**
#### Now we move to training. This is where the model learns patterns from our training emails and checks its performance against the validation emails.
## **Train, Save, and Verify the Email Spam Identification Model**

#### ***1) Train the model with EarlyStopping and ModelCheckpoint***

In [ ]:
# Uses larger batch_size (128) for faster execution on large datasets.
history = email_model.fit(
    X_train_vectorized,
    y_train_encoded,
    epochs=10,
    batch_size=128, # I increased batch_size from 32 to 128 because your goal is building a
    # pipeline ready for large datasets.
    validation_data=(X_valid_vectorized, y_valid_encoded),
    callbacks=[
        # Stops training early if validation loss stops improving for 3 consecutive epochs.
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        ),
        # Automatically saves ONLY the best-performing model to disk during training.
        tf.keras.callbacks.ModelCheckpoint(
            filepath="going-thru-some-parts-of-email-spam-project/models/best_email_spam_model2.keras",
            monitor="val_loss",
            save_best_only=True,
            mode="min"
        )
    ]
)

#### ***2) Save and reload the TextVectorization layer (with Windows UTF-8 fix)***

In [ ]:
import builtins
import io
from contextlib import contextmanager


@contextmanager
def force_utf8_open():
    original_builtin_open = builtins.open
    original_open = io.open

    def utf8_open(file, mode="r", *args, **kwargs):
        if "b" not in mode:
            kwargs.setdefault("encoding", "utf-8")
        return original_open(file, mode, *args, **kwargs)

    builtins.open = utf8_open
    try:
        yield
    finally:
        # Always restores Python's default file opener, even if an error occurs.
        builtins.open = original_builtin_open

# Wraps the adapted TextVectorization layer inside a Sequential model so Keras can save it.
vectorizer_model = tf.keras.Sequential([
    tf.keras.Input(shape=(), dtype=tf.string),
    text_vectorizer
])

# Saves the vectorizer safely with UTF-8 encoding (preserves emojis and special characters).
with force_utf8_open():
    vectorizer_model.save(
        "going-thru-some-parts-of-email-spam-project/models/email_text_vectorizer2.keras"
    )
    print("TextVectorization model saved successfully!")

#### ***3) Load both saved artifacts to verify standalone inference***

In [ ]:
# Loads the best model saved by ModelCheckpoint in Step 1.
loaded_email_model = tf.keras.models.load_model(
    "going-thru-some-parts-of-email-spam-project/models/best_email_spam_model2.keras"
)

# Loads the saved vectorizer using our UTF-8 helper.
with force_utf8_open():
    loaded_vectorizer_model = tf.keras.models.load_model(
        "going-thru-some-parts-of-email-spam-project/models/email_text_vectorizer2.keras"
    )
    print("Both model and vectorizer loaded successfully!")


#### ***4) Test the complete saved pipeline on sample emails (Batch Prediction)***

In [ ]:
def test_saved_pipeline(raw_emails, vectorizer, model, threshold=0.5):
    """
    Runs end-to-end prediction on raw email strings using our full cleaning + model pipeline.
    """
    # Applies our custom clean_email() function from Stage 3 to each incoming email.
    cleaned_emails = [clean_email(email) for email in raw_emails]
    
    # Converts cleaned strings into padded integer sequences.
    vectorized = vectorizer(tf.constant(cleaned_emails))
    
    # Predicts spam probabilities for all emails at once in a single batch.
    probabilities = model.predict(vectorized, verbose=0).flatten()
    
    # Prints readable results for each test email.
    for email_text, prob in zip(raw_emails, probabilities):
        label = "spam" if prob >= threshold else "not_spam"
        print(f"Email: {email_text[:60]}...")
        print(f"Spam probability: {prob:.4f} | Predicted label: {label}")
        print("-" * 80)

## **Tests both a spam sample and a legitimate sample together in one clean call.**

In [ ]:
sample_emails = [
    "Congratulations! You have won a prize. Claim your reward now!",
    "Hi John, please find attached the meeting notes from our discussion yesterday. Let me know if you have any questions."
]

test_saved_pipeline(sample_emails, loaded_vectorizer_model, loaded_email_model)


# **Model Evaluation.**
#### ***Evaluate on the test dataset***

In [ ]:
test_loss, test_accuracy, test_precision, test_recall = email_model.evaluate(
    X_test_vectorized,
    y_test_encoded
)

print(f"Test loss:      {test_loss:.4f}")
print(f"Test accuracy:  {test_accuracy:.4f}")
print(f"Test precision: {test_precision:.4f}")
print(f"Test recall:    {test_recall:.4f}")